# Energy Sector Stock Direction Prediction
## Quant-Grade ML Approaches: LASSO-LSTM · LightGBM · TFT · VSN+LSTM

**Target:** Binary classification — next-day price direction (up/down)  
**Features:** 174 log-returns (macro indices + 166 energy stocks)  
**Split:** Train 2016–2023 | Val 2024 | Test 2025  

### Models Implemented
1. **LASSO-LSTM** — LASSO feature selection → LSTM (handles multicollinearity)
2. **LightGBM + Walk-Forward** — Gradient boosting with expanding window & SHAP
3. **Temporal Fusion Transformer (TFT)** — DeepMind's attention-based temporal model
4. **VSN + LSTM** — Variable Selection Network feeding into LSTM (highest Sharpe in literature)

### Metrics
Accuracy · F1 · AUC-ROC · Directional Accuracy · Pseudo-Sharpe

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

# Sklearn
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.pipeline import Pipeline

# LightGBM
import lightgbm as lgb

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# PyTorch Forecasting (TFT)
import pytorch_lightning as pl
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.metrics import CrossEntropy

# Display settings
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='darkgrid', palette='muted')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'LightGBM: {lgb.__version__}')

## 1. Data Loading & EDA

In [ ]:
# ── Load all splits ───────────────────────────────────────────────────────────
BASE = Path('.')  # Update path if running outside project directory

X_train = pd.read_csv(BASE / 'X_train20162023.csv', index_col='Date', parse_dates=True)
y_train = pd.read_csv(BASE / 'y_train120162023.csv', index_col='Date', parse_dates=True)

X_val   = pd.read_csv(BASE / 'X_val2024.csv',       index_col='Date', parse_dates=True)
y_val   = pd.read_csv(BASE / 'y_val2024.csv',        index_col='Date', parse_dates=True)

X_test  = pd.read_csv(BASE / 'X_test2025.csv',      index_col='Date', parse_dates=True)
y_test  = pd.read_csv(BASE / 'y_test2025.csv',       index_col='Date', parse_dates=True)

TARGET = 'Target_Next_Day_Direction'

print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'\nClass balance (train):')
print(y_train[TARGET].value_counts(normalize=True).round(3))

In [ ]:
# ── EDA: Class distribution & feature correlation heatmap ─────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, y) in zip(axes, [('Train', y_train), ('Val', y_val), ('Test', y_test)]):
    counts = y[TARGET].value_counts()
    ax.bar(['Down (0)', 'Up (1)'], counts.values, color=['#e74c3c', '#2ecc71'], alpha=0.85)
    ax.set_title(f'{name} — Class Balance')
    ax.set_ylabel('Count')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 5, f'{v/sum(counts.values):.1%}', ha='center', fontweight='bold')

plt.suptitle('Target Distribution Across Splits', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Macro feature correlation with target ────────────────────────────────────
macro_cols = X_train.columns[:8].tolist()  # VIX, OVX, Oil, NatGas, DXY, SP500, Nikkei, FTSE

combined_train = X_train[macro_cols].copy()
combined_train['Target'] = y_train[TARGET]
corr = combined_train.corr()['Target'].drop('Target').sort_values()

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#e74c3c' if c < 0 else '#2ecc71' for c in corr.values]
corr.plot(kind='barh', ax=ax, color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Macro Feature Correlation with Next-Day Direction (Train)', fontweight='bold')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

In [ ]:
# ── Preprocessing helpers ─────────────────────────────────────────────────────
def impute_and_scale(X_tr, X_v, X_te):
    """Fill NaNs with column median (train stats) and StandardScale."""
    medians = X_tr.median()
    X_tr = X_tr.fillna(medians)
    X_v  = X_v.fillna(medians)
    X_te = X_te.fillna(medians)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_v_s  = scaler.transform(X_v)
    X_te_s = scaler.transform(X_te)
    return X_tr_s, X_v_s, X_te_s, scaler


def make_sequences(X, y, seq_len=20):
    """Create sliding-window sequences for LSTM-style models."""
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i - seq_len:i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def evaluate(name, y_true, y_pred, y_prob, returns=None):
    """Compute standard classification + pseudo-Sharpe metrics."""
    acc  = accuracy_score(y_true, y_pred)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    auc  = roc_auc_score(y_true, y_prob)
    da   = acc  # directional accuracy == accuracy for binary

    # Pseudo-Sharpe: treat prediction confidence as signal strength
    sharpe = np.nan
    if returns is not None:
        signal = np.where(y_pred == 1, 1, -1)
        strat_ret = signal * np.array(returns)
        if strat_ret.std() > 0:
            sharpe = np.sqrt(252) * strat_ret.mean() / strat_ret.std()

    print(f'\n{'─'*45}')
    print(f'  {name}')
    print(f'{'─'*45}')
    print(f'  Accuracy:           {acc:.4f}')
    print(f'  F1 Score:           {f1:.4f}')
    print(f'  AUC-ROC:            {auc:.4f}')
    print(f'  Directional Acc:    {da:.4f}')
    if not np.isnan(sharpe):
        print(f'  Pseudo-Sharpe:      {sharpe:.4f}')

    return {'model': name, 'accuracy': acc, 'f1': f1, 'auc': auc,
            'directional_acc': da, 'pseudo_sharpe': sharpe}


results = []  # Collect all model results
print('Helpers defined.')

## 2. Model 1: LASSO-LSTM
> LASSO performs sparse feature selection, then LSTM captures temporal dependencies on the reduced feature set. Addresses multicollinearity between the 166 correlated energy stocks.

In [ ]:
# ── Step 1: LASSO feature selection ──────────────────────────────────────────
X_tr_s, X_v_s, X_te_s, scaler_lasso = impute_and_scale(X_train, X_val, X_test)
y_tr = y_train[TARGET].values
y_v  = y_val[TARGET].values
y_te = y_test[TARGET].values

print('Fitting LassoCV for feature selection...')
lasso = LassoCV(cv=5, max_iter=5000, n_jobs=-1, random_state=SEED)
lasso.fit(X_tr_s, y_tr)

selected_mask = lasso.coef_ != 0
selected_features = X_train.columns[selected_mask].tolist()
print(f'Selected {selected_mask.sum()} / {X_train.shape[1]} features (alpha={lasso.alpha_:.5f})')

# Reduce to selected features
X_tr_l = X_tr_s[:, selected_mask]
X_v_l  = X_v_s[:, selected_mask]
X_te_l = X_te_s[:, selected_mask]

print(f'\nTop 10 selected features by |coef|:')
coef_df = pd.Series(np.abs(lasso.coef_[selected_mask]), index=selected_features).sort_values(ascending=False)
print(coef_df.head(10).to_string())

In [ ]:
# ── Step 2: Build sequences ───────────────────────────────────────────────────
SEQ_LEN = 20  # 20 trading days lookback (~1 month)

# Concatenate train+val for sequence boundary handling
X_all_l = np.vstack([X_tr_l, X_v_l, X_te_l])
y_all   = np.concatenate([y_tr, y_v, y_te])

n_train, n_val, n_test = len(y_tr), len(y_v), len(y_te)

Xs, ys = make_sequences(X_all_l, y_all, SEQ_LEN)

# Adjust split boundaries
tr_end  = n_train - SEQ_LEN
v_end   = tr_end + n_val

Xtr, ytr = Xs[:tr_end], ys[:tr_end]
Xv,  yv  = Xs[tr_end:v_end], ys[tr_end:v_end]
Xte, yte = Xs[v_end:], ys[v_end:]

print(f'Sequence shapes — Train: {Xtr.shape} | Val: {Xv.shape} | Test: {Xte.shape}')

In [ ]:
# ── Step 3: Define LSTM model ─────────────────────────────────────────────────
class LassoLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.dropout(out[:, -1, :])  # Last timestep
        return self.fc(out).squeeze(1)


def train_lstm(model, Xtr, ytr, Xv, yv, epochs=50, batch_size=64, lr=1e-3, patience=10):
    """Train with early stopping on validation loss."""
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

    tr_ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=False)

    Xv_t = torch.from_numpy(Xv).to(DEVICE)
    yv_t = torch.from_numpy(yv).to(DEVICE)

    best_val_loss, best_state, no_improve = np.inf, None, 0
    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        for Xb, yb in tr_dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred = model(Xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xv_t), yv_t).item()

        scheduler.step(val_loss)
        train_losses.append(ep_loss / len(tr_dl))
        val_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch}')
                break

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_loss:.4f}')

    model.load_state_dict(best_state)
    return model, train_losses, val_losses


print('LASSO-LSTM architecture defined.')

In [ ]:
# ── Step 4: Train & evaluate LASSO-LSTM ──────────────────────────────────────
n_features_lasso = Xtr.shape[2]
lasso_lstm = LassoLSTM(input_dim=n_features_lasso, hidden_dim=64, num_layers=2, dropout=0.3).to(DEVICE)
print(f'LASSO-LSTM params: {sum(p.numel() for p in lasso_lstm.parameters()):,}')
print(f'Input dim (selected features): {n_features_lasso}\n')

lasso_lstm, tr_losses, v_losses = train_lstm(
    lasso_lstm, Xtr, ytr, Xv, yv, epochs=100, batch_size=64, lr=1e-3, patience=15
)

# Predict on test
lasso_lstm.eval()
with torch.no_grad():
    test_prob_ll = lasso_lstm(torch.from_numpy(Xte).to(DEVICE)).cpu().numpy()
test_pred_ll = (test_prob_ll >= 0.5).astype(int)

# Also predict val for comparison
with torch.no_grad():
    val_prob_ll = lasso_lstm(torch.from_numpy(Xv).to(DEVICE)).cpu().numpy()
val_pred_ll = (val_prob_ll >= 0.5).astype(int)

res_ll = evaluate('LASSO-LSTM (Test 2025)', yte, test_pred_ll, test_prob_ll)
results.append(res_ll)

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(tr_losses, label='Train Loss', color='#3498db')
ax.plot(v_losses,  label='Val Loss',   color='#e74c3c')
ax.set_title('LASSO-LSTM: Training Curve', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.legend()
plt.tight_layout()
plt.show()

# LASSO feature importance
fig, ax = plt.subplots(figsize=(10, 5))
top20 = coef_df.head(20)
top20.plot(kind='barh', ax=ax, color='#2980b9', alpha=0.85)
ax.set_title('LASSO Selected Features — Top 20 by |Coefficient|', fontweight='bold')
ax.set_xlabel('|LASSO Coefficient|')
plt.tight_layout()
plt.show()

## 3. Model 2: LightGBM with Walk-Forward Validation + SHAP
> Gradient boosting with expanding window retraining. The quant-industry workhorse for tabular financial data. SHAP provides interpretable feature attribution — critical for validating economic intuition.

In [ ]:
# ── LightGBM: Walk-Forward Expanding Window ───────────────────────────────────
# Combine train + val for walk-forward, test on 2025

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])[TARGET]

# Impute NaNs
medians_lgb = X_train.median()
X_trainval_imp = X_trainval.fillna(medians_lgb)
X_test_imp     = X_test.fillna(medians_lgb)

# Walk-forward: retrain every 21 days (monthly) with expanding window
STEP = 21  # Monthly rebalance

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_estimators': 500,
    'random_state': SEED,
    'verbose': -1,
    'n_jobs': -1
}

print('Running walk-forward LightGBM...')

# For test set, train on all train+val, predict test
lgb_model_final = lgb.LGBMClassifier(**lgb_params)

# Val-set walk-forward for honest evaluation
val_probs_wf = np.zeros(len(X_val))
X_tr_imp = X_train.fillna(medians_lgb)
X_v_imp  = X_val.fillna(medians_lgb)
y_tr_s   = y_train[TARGET]
y_v_s    = y_val[TARGET]

for start in range(0, len(X_val), STEP):
    end = min(start + STEP, len(X_val))

    # Expanding window: all train + val up to 'start'
    X_expand = pd.concat([X_tr_imp, X_v_imp.iloc[:start]]) if start > 0 else X_tr_imp
    y_expand = pd.concat([y_tr_s, y_v_s.iloc[:start]])     if start > 0 else y_tr_s

    m = lgb.LGBMClassifier(**lgb_params)
    m.fit(
        X_expand, y_expand,
        eval_set=[(X_v_imp.iloc[start:end], y_v_s.iloc[start:end])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)]
    )
    val_probs_wf[start:end] = m.predict_proba(X_v_imp.iloc[start:end])[:, 1]

val_preds_wf = (val_probs_wf >= 0.5).astype(int)
res_lgb_val = evaluate('LightGBM Walk-Forward (Val 2024)', y_v_s.values, val_preds_wf, val_probs_wf)

# Final model: train on all train+val, predict test
print('\nTraining final LightGBM on train+val for test evaluation...')
lgb_model_final.fit(
    X_trainval_imp, y_trainval,
    callbacks=[lgb.log_evaluation(period=-1)]
)
test_prob_lgb  = lgb_model_final.predict_proba(X_test_imp)[:, 1]
test_pred_lgb  = (test_prob_lgb >= 0.5).astype(int)

res_lgb = evaluate('LightGBM (Test 2025)', y_test[TARGET].values, test_pred_lgb, test_prob_lgb)
results.append(res_lgb)

In [ ]:
# ── SHAP Feature Importance ───────────────────────────────────────────────────
try:
    import shap
    print('Computing SHAP values...')
    explainer = shap.TreeExplainer(lgb_model_final)
    shap_values = explainer.shap_values(X_test_imp)

    # For binary: shap_values is list [class0, class1] or 3D array
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    mean_shap = pd.Series(
        np.abs(sv).mean(axis=0),
        index=X_test_imp.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    mean_shap.head(25).plot(kind='barh', ax=ax, color='#8e44ad', alpha=0.85)
    ax.invert_yaxis()
    ax.set_title('LightGBM SHAP Feature Importance — Top 25 (Test 2025)', fontweight='bold')
    ax.set_xlabel('Mean |SHAP Value|')
    plt.tight_layout()
    plt.show()

except ImportError:
    print('SHAP not installed. Showing built-in feature importance instead.')
    feat_imp = pd.Series(
        lgb_model_final.feature_importances_,
        index=X_trainval_imp.columns
    ).sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 6))
    feat_imp.head(25).plot(kind='barh', ax=ax, color='#8e44ad', alpha=0.85)
    ax.invert_yaxis()
    ax.set_title('LightGBM Feature Importance — Top 25', fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Model 3: Temporal Fusion Transformer (TFT)
> DeepMind's purpose-built architecture for multi-horizon time series. Combines variable selection networks, LSTM encoders, and multi-head attention. Best-in-class for structured temporal data.

In [ ]:
# ── TFT: Data preparation ─────────────────────────────────────────────────────
# TFT requires a specific long-format DataFrame with time_idx and group_id

def prepare_tft_data(X_tr, y_tr, X_v, y_v, X_te, y_te, max_encoder_length=20):
    """Build combined DataFrame for pytorch-forecasting TimeSeriesDataSet."""
    medians = X_tr.median()

    dfs = []
    for X, y, split in [(X_tr, y_tr, 'train'), (X_v, y_v, 'val'), (X_te, y_te, 'test')]:
        df = X.fillna(medians).copy()
        df['target'] = y[TARGET].values
        df['split']  = split
        df['group']  = 'energy_sector'  # Single time series group
        dfs.append(df)

    combined = pd.concat(dfs)
    combined = combined.reset_index(drop=False)
    combined['time_idx'] = range(len(combined))
    combined['target']   = combined['target'].astype(float)

    # Clip extreme values to avoid gradient issues
    feat_cols = X_tr.columns.tolist()
    for col in feat_cols:
        combined[col] = combined[col].clip(-10, 10)

    return combined, feat_cols


print('Preparing TFT dataset...')
tft_df, feat_cols = prepare_tft_data(X_train, y_train, X_val, y_val, X_test, y_test)

MAX_ENCODER = 20
MAX_PRED    = 1
N_TRAIN     = len(X_train)
N_VAL       = len(X_val)

# Use a subset of features for TFT (top correlated with target to limit complexity)
# This keeps training tractable; use all for production
ALL_FEATS_CORR = (
    pd.concat([X_train.fillna(X_train.median()), y_train], axis=1)
    .corr()[TARGET]
    .abs()
    .drop(TARGET)
    .sort_values(ascending=False)
)
TOP_FEATS = ALL_FEATS_CORR.head(40).index.tolist()  # Top 40 features
print(f'Using top {len(TOP_FEATS)} features for TFT (ranked by |correlation| with target)')

training = TimeSeriesDataSet(
    tft_df[tft_df['time_idx'] < N_TRAIN],
    time_idx='time_idx',
    target='target',
    group_ids=['group'],
    min_encoder_length=MAX_ENCODER,
    max_encoder_length=MAX_ENCODER,
    min_prediction_length=MAX_PRED,
    max_prediction_length=MAX_PRED,
    time_varying_unknown_reals=TOP_FEATS,
    target_normalizer=None,
    add_relative_time_idx=True,
    add_target_scales=False,
)

validation = TimeSeriesDataSet.from_dataset(
    training,
    tft_df[tft_df['time_idx'] < N_TRAIN + N_VAL],
    predict=True,
    stop_randomization=True
)

train_dl_tft = training.to_dataloader(train=True,  batch_size=64, num_workers=0)
val_dl_tft   = validation.to_dataloader(train=False, batch_size=64, num_workers=0)

print('TFT datasets ready.')

In [ ]:
# ── TFT Model Definition & Training ──────────────────────────────────────────
tft_model = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=1e-3,
    hidden_size=32,
    attention_head_size=2,
    dropout=0.2,
    hidden_continuous_size=16,
    loss=CrossEntropy(),
    optimizer='Adam',
    reduce_on_plateau_patience=5,
)

print(f'TFT parameters: {tft_model.size():,}')

trainer = pl.Trainer(
    max_epochs=30,
    accelerator='gpu' if torch.cuda.is_available() else 'cpu',
    enable_progress_bar=True,
    gradient_clip_val=0.1,
    callbacks=[
        pl.callbacks.EarlyStopping(monitor='val_loss', patience=8, mode='min'),
        pl.callbacks.ModelCheckpoint(monitor='val_loss', mode='min', save_top_k=1)
    ],
    logger=False,
    enable_model_summary=False,
)

print('Training TFT...')
trainer.fit(tft_model, train_dataloaders=train_dl_tft, val_dataloaders=val_dl_tft)
print('TFT training complete.')

In [ ]:
# ── TFT: Predict on validation set ───────────────────────────────────────────
best_tft = TemporalFusionTransformer.load_from_checkpoint(trainer.checkpoint_callback.best_model_path)
predictions = best_tft.predict(val_dl_tft, mode='raw', return_x=True)

# CrossEntropy output: probabilities for class 1
pred_raw = predictions.output.prediction.squeeze().cpu().numpy()
# If output is 2D (batch, n_classes), take class-1 prob
if pred_raw.ndim == 2:
    tft_prob = pred_raw[:, 1]
else:
    tft_prob = torch.sigmoid(torch.tensor(pred_raw)).numpy()

tft_pred = (tft_prob >= 0.5).astype(int)
y_val_tft = y_val[TARGET].values[MAX_ENCODER:MAX_ENCODER + len(tft_prob)]

res_tft = evaluate('TFT (Val 2024)', y_val_tft, tft_pred, tft_prob)
results.append(res_tft)

In [ ]:
# ── TFT: Variable Importance ──────────────────────────────────────────────────
try:
    interpretation = best_tft.interpret_output(predictions.output, reduction='sum')
    best_tft.plot_interpretation(interpretation)
    plt.suptitle('TFT Variable Importance', fontweight='bold')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Interpretation plot unavailable: {e}')

## 5. Model 4: VSN + LSTM
> Variable Selection Network (VSN) learns sparse, attention-weighted feature gates at each timestep, then feeds the selected representation into an LSTM. Highest overall Sharpe ratio in the literature for energy equity direction prediction.

In [ ]:
# ── VSN Architecture ──────────────────────────────────────────────────────────
class GatedResidualNetwork(nn.Module):
    """Core GRN block used in VSN and TFT architectures."""
    def __init__(self, input_dim, hidden_dim, output_dim, dropout=0.1):
        super().__init__()
        self.fc1     = nn.Linear(input_dim, hidden_dim)
        self.fc2     = nn.Linear(hidden_dim, output_dim)
        self.gate    = nn.Linear(input_dim, output_dim)
        self.norm    = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)
        self.act     = nn.ELU()
        self.proj    = nn.Linear(input_dim, output_dim) if input_dim != output_dim else nn.Identity()

    def forward(self, x):
        h   = self.act(self.fc1(x))
        h   = self.dropout(self.fc2(h))
        g   = torch.sigmoid(self.gate(x))
        out = self.norm(g * h + self.proj(x))
        return out


class VariableSelectionNetwork(nn.Module):
    """VSN: learns per-feature soft gates → weighted combination."""
    def __init__(self, input_dim, hidden_dim, dropout=0.1):
        super().__init__()
        self.hidden_dim   = hidden_dim
        self.feature_nets = nn.ModuleList([
            GatedResidualNetwork(1, hidden_dim, hidden_dim, dropout)
            for _ in range(input_dim)
        ])
        self.softmax_net  = GatedResidualNetwork(input_dim, hidden_dim, input_dim, dropout)
        self.softmax      = nn.Softmax(dim=-1)

    def forward(self, x):
        # x: (batch, seq_len, input_dim)
        B, T, D = x.shape
        # Per-feature transformations
        xi = torch.stack([self.feature_nets[i](x[:, :, i:i+1]) for i in range(D)], dim=-2)
        # (B, T, D, hidden_dim)
        # Variable selection weights
        weights = self.softmax(self.softmax_net(x))  # (B, T, D)
        # Weighted combination
        out = (weights.unsqueeze(-1) * xi).sum(dim=-2)  # (B, T, hidden_dim)
        return out, weights


class VSN_LSTM(nn.Module):
    """VSN + LSTM: Variable selection followed by temporal LSTM encoder."""
    def __init__(self, input_dim, vsn_hidden=32, lstm_hidden=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.vsn  = VariableSelectionNetwork(input_dim, vsn_hidden, dropout)
        self.lstm = nn.LSTM(vsn_hidden, lstm_hidden, num_layers,
                            batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, 32),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        vsn_out, weights = self.vsn(x)       # (B, T, vsn_hidden)
        lstm_out, _      = self.lstm(vsn_out) # (B, T, lstm_hidden)
        return self.head(lstm_out[:, -1, :]).squeeze(1), weights


print('VSN+LSTM architecture defined.')

In [ ]:
# ── VSN+LSTM: Prepare data (use all features — VSN handles selection) ─────────
X_tr_s2, X_v_s2, X_te_s2, scaler2 = impute_and_scale(X_train, X_val, X_test)

X_all2 = np.vstack([X_tr_s2, X_v_s2, X_te_s2]).astype(np.float32)
y_all2 = np.concatenate([y_tr, y_v, y_te])

Xs2, ys2 = make_sequences(X_all2, y_all2, SEQ_LEN)

Xtr2 = Xs2[:tr_end]
ytr2 = ys2[:tr_end]
Xv2  = Xs2[tr_end:v_end]
yv2  = ys2[tr_end:v_end]
Xte2 = Xs2[v_end:]
yte2 = ys2[v_end:]

print(f'Shapes — Train: {Xtr2.shape} | Val: {Xv2.shape} | Test: {Xte2.shape}')
n_features_all = Xtr2.shape[2]

In [ ]:
# ── VSN+LSTM: Training ────────────────────────────────────────────────────────
def train_vsn_lstm(model, Xtr, ytr, Xv, yv, epochs=80, batch_size=64, lr=5e-4, patience=15):
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.BCELoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    tr_ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr))
    tr_dl = DataLoader(tr_ds, batch_size=batch_size, shuffle=False)

    Xv_t = torch.from_numpy(Xv).to(DEVICE)
    yv_t = torch.from_numpy(yv).to(DEVICE)

    best_val_loss, best_state, no_improve = np.inf, None, 0
    tr_losses, vl_losses = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        for Xb, yb in tr_dl:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred, _ = model(Xb)
            loss = criterion(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ep_loss += loss.item()

        model.eval()
        with torch.no_grad():
            val_pred, _ = model(Xv_t)
            val_loss = criterion(val_pred, yv_t).item()

        scheduler.step()
        tr_losses.append(ep_loss / len(tr_dl))
        vl_losses.append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch}')
                break

        if epoch % 10 == 0:
            print(f'  Epoch {epoch:3d} | Train Loss: {tr_losses[-1]:.4f} | Val Loss: {val_loss:.4f}')

    model.load_state_dict(best_state)
    return model, tr_losses, vl_losses


vsn_lstm = VSN_LSTM(
    input_dim=n_features_all, vsn_hidden=32, lstm_hidden=64,
    num_layers=2, dropout=0.3
).to(DEVICE)

print(f'VSN+LSTM params: {sum(p.numel() for p in vsn_lstm.parameters()):,}')
print(f'Input dim (all features): {n_features_all}\n')

vsn_lstm, vsn_tr_losses, vsn_vl_losses = train_vsn_lstm(
    vsn_lstm, Xtr2, ytr2, Xv2, yv2, epochs=100, batch_size=64, lr=5e-4, patience=15
)

# Evaluate
vsn_lstm.eval()
with torch.no_grad():
    test_prob_vsn, test_weights = vsn_lstm(torch.from_numpy(Xte2).to(DEVICE))
    test_prob_vsn = test_prob_vsn.cpu().numpy()
    test_weights  = test_weights.cpu().numpy()

test_pred_vsn = (test_prob_vsn >= 0.5).astype(int)
res_vsn = evaluate('VSN+LSTM (Test 2025)', yte2, test_pred_vsn, test_prob_vsn)
results.append(res_vsn)

In [ ]:
# ── VSN Feature Selection Visualization ──────────────────────────────────────
mean_weights = test_weights.mean(axis=(0, 1))  # (n_features,)
feature_importance_vsn = pd.Series(mean_weights, index=X_train.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# VSN top features
feature_importance_vsn.head(20).plot(kind='barh', ax=axes[0], color='#e67e22', alpha=0.85)
axes[0].invert_yaxis()
axes[0].set_title('VSN+LSTM: Top 20 Selected Features\n(Mean Attention Weight on Test Set)', fontweight='bold')
axes[0].set_xlabel('Mean Selection Weight')

# Training curve
axes[1].plot(vsn_tr_losses, label='Train Loss', color='#3498db')
axes[1].plot(vsn_vl_losses, label='Val Loss',   color='#e74c3c')
axes[1].set_title('VSN+LSTM: Training Curve', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('BCE Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Results Comparison

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results).set_index('model')
print('\n' + '='*65)
print('  FINAL MODEL COMPARISON')
print('='*65)
print(results_df[['accuracy', 'f1', 'auc', 'directional_acc', 'pseudo_sharpe']].to_string())
print('='*65)
print('\nBest model per metric:')
for col in ['accuracy', 'f1', 'auc', 'pseudo_sharpe']:
    valid = results_df[col].dropna()
    if len(valid):
        print(f'  {col:20s}: {valid.idxmax()}')

In [ ]:
# ── Visual Comparison ─────────────────────────────────────────────────────────
metrics = ['accuracy', 'f1', 'auc']
n_models = len(results_df)
x = np.arange(n_models)
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)

for ax, metric in zip(axes, metrics):
    vals = results_df[metric].values
    bars = ax.bar(x, vals, color=colors[:n_models], alpha=0.85, edgecolor='white', linewidth=1.2)
    ax.set_title(metric.upper().replace('_', ' '), fontweight='bold', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(results_df.index, rotation=20, ha='right', fontsize=9)
    ax.set_ylim(0.4, 1.0)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random baseline')
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                    f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    if metric == 'accuracy':
        ax.legend(fontsize=8)

plt.suptitle('Model Performance Comparison — Energy Sector Direction Prediction',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── ROC Curves ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

model_probs = {
    'LASSO-LSTM':  (yte,  test_prob_ll),
    'LightGBM':    (y_test[TARGET].values, test_prob_lgb),
    'VSN+LSTM':    (yte2, test_prob_vsn),
}

for (name, (y_true, y_prob)), color in zip(model_probs.items(), colors):
    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        auc_val = roc_auc_score(y_true, y_prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc_val:.3f})', color=color, linewidth=2)
    except Exception as e:
        print(f'ROC skipped for {name}: {e}')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — Test Set (2025)', fontweight='bold', fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Confusion Matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

model_preds = {
    'LASSO-LSTM':  (yte,  test_pred_ll),
    'LightGBM':    (y_test[TARGET].values, test_pred_lgb),
    'VSN+LSTM':    (yte2, test_pred_vsn),
}

for ax, (name, (y_true, y_pred)) in zip(axes, model_preds.items()):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Down', 'Up'], yticklabels=['Down', 'Up'])
    ax.set_title(f'{name}\nConfusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Test Set (2025)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 7. Strategy Backtest (Pseudo-Sharpe)
Using model predictions as long/short signals on a synthetic equal-weight energy portfolio.

In [ ]:
# ── Pseudo-portfolio backtest ─────────────────────────────────────────────────
# Use the macro Crude Oil log-return as a proxy for portfolio return
oil_test_ret = X_test['Crude_Oil_Futures_Log_Return'].fillna(0).values
oil_test_ret_vsn = oil_test_ret[SEQ_LEN:]  # Align with sequence predictions

buy_hold_cumret = np.cumprod(1 + oil_test_ret / 100)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(buy_hold_cumret, label='Buy & Hold (Crude Oil)', color='gray', linewidth=1.5, linestyle='--')

model_strategy = {
    'LASSO-LSTM':  (yte,  test_pred_ll,  oil_test_ret_vsn),
    'LightGBM':    (y_test[TARGET].values, test_pred_lgb, oil_test_ret),
    'VSN+LSTM':    (yte2, test_pred_vsn,  oil_test_ret_vsn),
}

for (name, (y_true, y_pred, ret)), color in zip(model_strategy.items(), colors):
    aligned = ret[:len(y_pred)]
    signal  = np.where(y_pred == 1, 1, -1)
    strat   = signal * aligned / 100
    cum_ret = np.cumprod(1 + strat)
    sharpe  = np.sqrt(252) * strat.mean() / strat.std() if strat.std() > 0 else np.nan
    ax.plot(cum_ret, label=f'{name} (Sharpe={sharpe:.2f})', color=color, linewidth=2)

ax.axhline(1.0, color='black', linewidth=0.8, linestyle=':')
ax.set_title('Simulated Long/Short Strategy — Test Period 2025\n(Using Crude Oil Log-Returns as Proxy)', fontweight='bold')
ax.set_xlabel('Trading Day')
ax.set_ylabel('Cumulative Return')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ── Final Summary ─────────────────────────────────────────────────────────────
print('\n' + '='*70)
print('  COMPLETE RESULTS SUMMARY')
print('='*70)
print(results_df.to_string())
print('='*70)

print("""
Architecture Notes:
──────────────────────────────────────────────────────────────────────
LASSO-LSTM   : LASSO shrinkage for feature selection → 2-layer LSTM
               Handles multicollinearity among 166 correlated tickers

LightGBM WF  : Expanding walk-forward gradient boosting (monthly refit)
               SHAP for economic interpretability; fastest inference

TFT          : TemporalFusionTransformer — VSN + LSTM encoder + 
               multi-head attention. Purpose-built for structured
               temporal data with interpretable attention weights

VSN+LSTM     : Custom Variable Selection Network → LSTM pipeline
               Learns per-timestep feature gates; highest Sharpe in
               energy equity direction literature
──────────────────────────────────────────────────────────────────────
""")